In [ ]:
import pandas as pd
import numpy as np
import torch
from TRGANLIB.TRGAN.TRGAN_train_load_modules import embeddings, load_model
from Scripts.data_preprocessing_sber import preprocessing_data_from_sber


In [2]:
data = preprocessing_data_from_sber(folder_path=r'Data\Sber\ditry_single')

Ищем CSV-файлы в: c:\Users\kiril\TRGAN\Data\Sber\ditry_single
Найдено файлов: 1
Файлы: ['c:\\Users\\kiril\\TRGAN\\Data\\Sber\\ditry_single\\transact1_2K.csv']
Загрузка и обработка файла: c:\Users\kiril\TRGAN\Data\Sber\ditry_single\transact1_2K.csv
Корректная обработка ADDRESS...
Статистика:
  Оффлайн транзакций: 70224 (21.7%)
  Онлайн транзакций: 253620 (78.3%)
  Физических адресов: 70218 (21.7%)
  Виртуальных транзакций: 253620 (78.3%)
Валидация логики обработки ADDRESS:
--------------------------------------------------
Онлайн транзакций: 253620
Онлайн транзакций, помеченных как виртуальные: 253620
Совпадение: 100.0%

Оффлайн транзакций: 70224
Оффлайн с физическими адресами: 70218
Покрытие: 100.0%

Распределение ADDRESS_TYPE:
ADDRESS_TYPE
VIRTUAL     253620
PHYSICAL     67241
OTHER         2977
MISSING          6
Name: count, dtype: int64
Итоговый датафрейм содержит 275259 строк и 36 колонок.


In [3]:
onehot_cols = ['PaymentSystem', 'OpType', 'DETAILEDCARDTYPE', 'TRANMETHOD', 'ISOWNTERMINAL', 'NAME', 'MCC', 'DEVICETYPE', 'CurrencyName', 'CITY', 'REGION', 'ISVIRTUALTRANSACTION']
cat_feat_names = ['ACCOUNT_ID', 'TERMINAL_CODE', 'CARD', 'TRANS_DETAIL']
num_feat_names = ['AMOUNT_EQ', 'HOUR', 'MINUTE', 'SECOND', 'EXCHANGE_RATA']
log1p_transform_cols = ['AMOUNT_EQ', 'EXCHANGE_RATA']  # если суммы имеют skewed распределение
date_feature = 'DATE'
time_feature = 'TRANS_TIME'
client_id = 'ACCOUNT_ID'
mcc_name = 'MCC'
latent_dim = {'onehot': 128, 'categorical': 16, 'numerical': 4, 'cv': 32}

In [4]:
data["REGION"].unique()

array(['VIRTUAL', 'Санкт-Петербург', 'UNKNOWN', 'обл Ленинградская',
       'обл Новгородская', 'обл Калининградская'], dtype=object)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 275259 entries, 122341 to 2923
Data columns (total 36 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   CustomerKey           275259 non-null  object        
 1   ID                    275259 non-null  object        
 2   TRANS_TIME            275259 non-null  datetime64[ns]
 3   AMOUNT_EQ             275259 non-null  float64       
 4   MCC                   275259 non-null  object        
 5   PAY_AMT               275259 non-null  float64       
 6   CurrencyName          275259 non-null  object        
 7   PaymentSystem         275259 non-null  object        
 8   TERMINAL_CODE         275259 non-null  object        
 9   ADDRESS               275259 non-null  object        
 10  DEVICETYPE            275259 non-null  object        
 11  TRANS_DETAIL          275259 non-null  object        
 12  NAME                  275259 non-null  object        
 13  O

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
directory = 'Pretrains_model_sber/Pretrains_V1/'  # папка с сохраненными моделями
experiment_id = 'sber'

In [7]:
len(data['TERMINAL_CODE'].unique())

52299

In [8]:
data[data['TERMINAL_CODE'] == 'NONE']

,CustomerKey,ID,TRANS_TIME,AMOUNT_EQ,MCC,PAY_AMT,CurrencyName,PaymentSystem,TERMINAL_CODE,ADDRESS,...,IS_PHYSICAL_LOCATION,ADDRESS_DETAIL_LEVEL,HAS_PHYSICAL_ADDRESS,ISVIRTUALTRANSACTION,OFFLINE_WITH_ADDRESS,ONLINE_VIRTUAL,EXCHANGE_RATA,HOUR,MINUTE,SECOND


In [9]:
X_emb, X_oh, cond_vector, synth_date, scaler_cat, scaler_onehot, scaler_num, cv_params, scaler, round_array = embeddings(
    data=data,  # можно передать None или заглушку, т.к. load=True
    cat_feat_names=cat_feat_names,
    num_feat_names=num_feat_names,
    onehot_cols=onehot_cols,
    date_feature=date_feature,
    time_feature=time_feature,
    client_id=client_id,
    latent_dim=latent_dim,
    device=device,
    load=True,  # ВАЖНО: загружаем предобученные!
    directory=directory
)

c:\Users\kiril\TRGAN\TRGANLIB\TRGAN\TRGAN_train_load_modules.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder_onehot.load_state_dict(torch.load(directory + 'oneh

In [10]:
# Загружаем обученные GAN модели
generator, supervisor, loss_array = load_model(
    latent_dim=latent_dim,
    dim_noise=20,  # размерность шума (должен совпадать с обучением)
    experiment_id=experiment_id,
    DIRECTORY=directory,
    DEVICE=device,
)

In [11]:
from TRGANLIB.TRGAN.TRGAN_main_V2 import sample, inverse_transform

# Сколько образцов сгенерировать
n_samples_data_len = len(X_emb[:])  # столько, сколько нужно

# Генерируем синтетические эмбеддинги и даты
synth_data, synth_date_gen, params = sample(
    n_samples=n_samples_data_len,
    generator=generator,
    supervisor=supervisor,
    noise_dim=20,  # размерность шума
    cond_vector=cond_vector,
    X_emb=X_emb,
    encoder=cv_params['encoder'],  # энкдер условного вектора
    data=data,  # нужен для генерации времени (можно передать исходные данные или заглушку)
    date_feature=date_feature,
    name_client_id=client_id,
    time_type='synth',  # или 'initial' если хочешь исходные времена
    cv_params=cv_params,
    device=device
)

In [12]:
synth_date_gen

,DATE
0,2017-05-01
1,2017-05-01
2,2017-05-01
3,2017-05-01
4,2017-05-01
...,...
275254,2019-02-28
275255,2019-02-28
275256,2019-03-06
275257,2019-03-06


In [13]:
X_emb1 = scaler.inverse_transform(X_emb)
synth_data = scaler.inverse_transform(synth_data)
print(synth_data)

synth_df = inverse_transform(synth_data, latent_dim, X_oh.columns, scaler_onehot, scaler_cat, scaler_num, cat_feat_names,
                             mcc_name, num_feat_names, True, synth_date_gen, time_feature, round_array, device=device)

[[ 0.9522657  -0.79981226  0.06871995 ... -0.97803247  0.99371475
  -0.9614396 ]
 [ 0.9121286  -0.23547015  0.44871208 ... -0.9810227   0.9951034
  -0.9618021 ]
 [-0.26736382 -0.88288283 -0.302753   ... -0.9795618   0.9952003
  -0.962707  ]
 ...
 [-0.866799    0.21512233 -0.8423375  ...  0.76575154  0.51115644
   0.52391815]
 [-0.11202759 -0.04373402 -0.3610958  ...  0.79448617  0.4444109
   0.44316745]
 [-0.358639    0.34202704 -0.32669553 ...  0.8190017   0.4855422
   0.5442736 ]]


In [14]:
synth_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275259 entries, 0 to 275258
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   PaymentSystem         275259 non-null  object        
 1   OpType                275259 non-null  object        
 2   DETAILEDCARDTYPE      275259 non-null  object        
 3   TRANMETHOD            275259 non-null  object        
 4   ISOWNTERMINAL         275259 non-null  object        
 5   NAME                  275259 non-null  object        
 6   MCC                   275259 non-null  object        
 7   DEVICETYPE            275259 non-null  object        
 8   CurrencyName          275259 non-null  object        
 9   CITY                  275259 non-null  object        
 10  REGION                275259 non-null  object        
 11  ISVIRTUALTRANSACTION  275259 non-null  object        
 12  ACCOUNT_ID            275259 non-null  object        
 13 

In [15]:
synth_df.to_csv('Data/Sber/Clear/transact_V1.csv', 
          index=False,           # Не записывать индексы
          sep=',',               # Разделитель
          encoding='utf-8',      # Кодировка
          header=True,           # Записывать заголовки
          na_rep='NULL')         # Замена NaN значений

In [16]:
n_samples_data_len = 10000  # столько, сколько нужно

# Генерируем синтетические эмбеддинги и даты
synth_data_small, synth_date_gen_small, params = sample(
    n_samples=n_samples_data_len,
    generator=generator,
    supervisor=supervisor,
    noise_dim=20,  # размерность шума
    cond_vector=cond_vector,
    X_emb=X_emb,
    encoder=cv_params['encoder'],  # энкдер условного вектора
    data=data,  # нужен для генерации времени (можно передать исходные данные или заглушку)
    date_feature=date_feature,
    name_client_id=client_id,
    time_type='synth',  # или 'initial' если хочешь исходные времена
    cv_params=cv_params,
    device=device
)

X_emb1 = scaler.inverse_transform(X_emb)
synth_data_small = scaler.inverse_transform(synth_data_small)

synth_small_df = inverse_transform(synth_data_small, latent_dim, X_oh.columns, scaler_onehot, scaler_cat, scaler_num, cat_feat_names,
                             mcc_name, num_feat_names, True, synth_date_gen_small, time_feature, round_array, device=device)

In [17]:
synth_small_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   PaymentSystem         10000 non-null  object        
 1   OpType                10000 non-null  object        
 2   DETAILEDCARDTYPE      10000 non-null  object        
 3   TRANMETHOD            10000 non-null  object        
 4   ISOWNTERMINAL         10000 non-null  object        
 5   NAME                  10000 non-null  object        
 6   MCC                   10000 non-null  object        
 7   DEVICETYPE            10000 non-null  object        
 8   CurrencyName          10000 non-null  object        
 9   CITY                  10000 non-null  object        
 10  REGION                10000 non-null  object        
 11  ISVIRTUALTRANSACTION  10000 non-null  object        
 12  ACCOUNT_ID            10000 non-null  object        
 13  TERMINAL_CODE    

In [18]:
synth_small_df.head()

,PaymentSystem,OpType,DETAILEDCARDTYPE,TRANMETHOD,ISOWNTERMINAL,NAME,MCC,DEVICETYPE,CurrencyName,CITY,REGION,ISVIRTUALTRANSACTION,ACCOUNT_ID,TERMINAL_CODE,CARD,TRANS_DETAIL,AMOUNT_EQ,EXCHANGE_RATA,DATE,TRANS_TIME
0,MasterCard,Оплата,MasterCard Unembossed,51,1,Оплата услуг через банкомат,4814,ATM,Рубль,Петергоф,Санкт-Петербург,0,286437,323785,2669621,"LENTA, SANKT-PETERBU, RU",33.73,0.994997,2017-05-01,00:00:00
1,MasterCard,Оплата,MasterCard World,51,0,Оплата товаров/услуг по карте,5411,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,286437,323785,2669621,"LENTA, SANKT-PETERBU, RU",33.53,0.994861,2017-05-01,00:00:00
2,MasterCard,Снятие наличных,MasterCard Unembossed,51,1,Выдача наличных ден. средств через банкомат,6011,ATM,Рубль,Санкт-Петербург,Санкт-Петербург,0,286437,323785,3005469,"LENTA, SANKT-PETERBU, RU",57.82,0.999107,2017-05-01,02:00:00
3,MasterCard,Оплата,MasterCard Unembossed,50,0,Оплата товаров/услуг по карте,5814,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,252076,10351824,2971872,"PEREKRESTOK, SANKT-PETERBU, RU",36.54,0.996253,2017-05-01,00:00:00
4,MasterCard,Оплата,MasterCard Unembossed,51,0,Оплата товаров/услуг по карте,5814,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,5524456,AC090097,2502195,"PEREKRESTOK, SANKT-PETERBU, RU",36.90,0.996818,2017-05-01,00:00:00


In [19]:
n_samples_data_len = 400000  # столько, сколько нужно

# Генерируем синтетические эмбеддинги и даты
synth_data_big, synth_date_gen_big, params = sample(
    n_samples=n_samples_data_len,
    generator=generator,
    supervisor=supervisor,
    noise_dim=20,  # размерность шума
    cond_vector=cond_vector,
    X_emb=X_emb,
    encoder=cv_params['encoder'],  # энкдер условного вектора
    data=data,  # нужен для генерации времени (можно передать исходные данные или заглушку)
    date_feature=date_feature,
    name_client_id=client_id,
    time_type='synth',  # или 'initial' если хочешь исходные времена
    cv_params=cv_params,
    device=device
)

X_emb1 = scaler.inverse_transform(X_emb)
synth_data_big = scaler.inverse_transform(synth_data_big)

synth_big_df = inverse_transform(synth_data_big, latent_dim, X_oh.columns, scaler_onehot, scaler_cat, scaler_num, cat_feat_names,
                             mcc_name, num_feat_names, True, synth_date_gen_big, time_feature, round_array, device=device)

In [20]:
synth_big_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400000 entries, 0 to 399999
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   PaymentSystem         400000 non-null  object        
 1   OpType                400000 non-null  object        
 2   DETAILEDCARDTYPE      400000 non-null  object        
 3   TRANMETHOD            400000 non-null  object        
 4   ISOWNTERMINAL         400000 non-null  object        
 5   NAME                  400000 non-null  object        
 6   MCC                   400000 non-null  object        
 7   DEVICETYPE            400000 non-null  object        
 8   CurrencyName          400000 non-null  object        
 9   CITY                  400000 non-null  object        
 10  REGION                400000 non-null  object        
 11  ISVIRTUALTRANSACTION  400000 non-null  object        
 12  ACCOUNT_ID            400000 non-null  object        
 13 

In [21]:
synth_big_df.head()

,PaymentSystem,OpType,DETAILEDCARDTYPE,TRANMETHOD,ISOWNTERMINAL,NAME,MCC,DEVICETYPE,CurrencyName,CITY,REGION,ISVIRTUALTRANSACTION,ACCOUNT_ID,TERMINAL_CODE,CARD,TRANS_DETAIL,AMOUNT_EQ,EXCHANGE_RATA,DATE,TRANS_TIME
0,MasterCard,Оплата,MasterCard Unembossed,51,0,Оплата товаров/услуг по карте,5411,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,2130791,10468841,2894080,"PYATEROCHKA 7000, MURINO, RU",196.90,0.999878,2017-05-02,11:12:12
1,MasterCard,Оплата,MasterCard Unembossed,71,0,Оплата товаров/услуг по карте,5814,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,8401808,Y3200059,2501038,"PYATEROCHKA 263, LOMONOSOV, RU",145.59,0.999793,2018-08-02,10:07:07
2,MasterCard,Оплата,MasterCard World,71,0,Оплата товаров/услуг по карте,5814,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,1428812,77122099,2366891,"LUKOIL.AZK 83 47S, SESTRORECK, RU",138.40,0.999718,2017-06-30,10:06:06
3,MasterCard,Оплата,MasterCard Unembossed,51,0,Оплата товаров/услуг по карте,5812,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,2329561,10335360,2165285,"ZANEVSKIY 7 OVOSHCHI, SANKT-PETERBU, RU",315.33,0.999929,2017-05-29,14:22:22
4,БАНК,Оплата,Дошкольная карта,901,1,Оплата товаров/услуг по карте,5641,POS,Рубль,Санкт-Петербург,Санкт-Петербург,0,7333575,10238339,3009481,"DIXY-47206, STARAYA, RU",275.36,0.999920,2017-09-15,13:19:19
